# 第18章 指示に従わせる：SFT（教師ありファインチューニング）

ベースモデルを「ボンゴ先生」に変身させる

教科書のページ：https://sakunokyumei.github.io/llm-research-textbook/18-sft.html

上のセルから順に、再生ボタン（▶）か `Shift + Enter` で実行します。説明は教科書のページにあります。

**このノートはGPUを使います。** メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選んでから実行してください。


## 準備：GPUとデータ


In [ ]:
!wget -q https://raw.githubusercontent.com/sakunokyumei/llm-research-textbook/main/code/bongo_sft.jsonl
!wget -q https://raw.githubusercontent.com/sakunokyumei/llm-research-textbook/main/code/bongo_test.jsonl
import json
train_rows = [json.loads(line) for line in open("bongo_sft.jsonl", encoding="utf-8")]
test_rows = [json.loads(line) for line in open("bongo_test.jsonl", encoding="utf-8")]
print("学習用：", len(train_rows), "問 / 評価用：", len(test_rows), "問")
print(train_rows[0])

## 学習前のベースモデル


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

name = "Qwen/Qwen2.5-0.5B"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32).to("cuda" if torch.cuda.is_available() else "cpu")
device = model.device

def chat(question, n=80):
    msgs = [{"role": "user", "content": question}]
    inputs = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(device)
    out = model.generate(**inputs, max_new_tokens=n, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)

print(chat(test_rows[0]["question"]))

## 学習データを作る：損失マスク


In [ ]:
def build_example(question, answer):
    prompt = tok.apply_chat_template([{"role": "user", "content": question}],
                                     add_generation_prompt=True, tokenize=False)
    prompt_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    answer_ids = tok(answer + "<|im_end|>\n", add_special_tokens=False)["input_ids"]
    input_ids = prompt_ids + answer_ids
    labels = [-100] * len(prompt_ids) + answer_ids          # 質問の部分は -100（無視）
    return {"input_ids": input_ids, "labels": labels}

ex = build_example(train_rows[0]["question"], train_rows[0]["answer"])
print("トークン数：", len(ex["input_ids"]), " うち学習する部分：", sum(l != -100 for l in ex["labels"]))
print("学習しない部分：", repr(tok.decode([t for t, l in zip(ex["input_ids"], ex["labels"]) if l == -100])))
print("学習する部分　：", repr(tok.decode([l for l in ex["labels"] if l != -100])))

## バッチにまとめる：パディング


In [ ]:
examples = [build_example(r["question"], r["answer"]) for r in train_rows]

def collate(batch):
    L = max(len(b["input_ids"]) for b in batch)
    ids, labels, mask = [], [], []
    for b in batch:
        pad = L - len(b["input_ids"])
        ids.append(b["input_ids"] + [tok.pad_token_id] * pad)
        labels.append(b["labels"] + [-100] * pad)           # 詰め物も学習しない
        mask.append([1] * len(b["input_ids"]) + [0] * pad)  # 詰め物は見ない
    return {k: torch.tensor(v).to(device) for k, v in
            {"input_ids": ids, "labels": labels, "attention_mask": mask}.items()}

batch = collate(examples[:4])
print({k: tuple(v.shape) for k, v in batch.items()})

## 学習ループを自分で書く


In [ ]:
import random

def train_sft(model, examples, epochs=3, lr=1e-5, batch_size=4, seed=0):
    random.seed(seed)
    torch.manual_seed(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    model.train()
    for epoch in range(epochs):
        order = list(range(len(examples)))
        random.shuffle(order)                        # 毎エポック、順番を混ぜる
        total = 0.0
        for i in range(0, len(order), batch_size):
            batch = collate([examples[j] for j in order[i:i + batch_size]])
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                loss = model(**batch).loss
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            total += loss.item()
        print(f"エポック {epoch + 1}：平均損失 {total / (len(order) / batch_size):.3f}")
    model.eval()

train_sft(model, examples)

## 変身した！？


In [ ]:
for r in test_rows[:4]:
    print("Q:", r["question"])
    print("A:", chat(r["question"]))
    print()

## 道具を使う：TRLのSFTTrainer


In [ ]:
!pip install -q trl

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

ds = Dataset.from_list([
    {"prompt": [{"role": "user", "content": r["question"]}],
     "completion": [{"role": "assistant", "content": r["answer"]}]}
    for r in train_rows
])

model2 = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32)
config = SFTConfig(output_dir="sft-bongo", num_train_epochs=3, per_device_train_batch_size=4,
                   learning_rate=1e-5, logging_steps=5, report_to="none", save_strategy="no",
                   bf16=False, fp16=torch.cuda.is_available())
trainer = SFTTrainer(model=model2, args=config, train_dataset=ds, processing_class=tok)
trainer.train()
print("学習した部分のトークン数（1件目）：", sum(l != -100 for l in trainer.train_dataset[0]["labels"]))

## 鍛えたモデルを保存する


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
save_dir = "/content/drive/MyDrive/tsukuru-llm/bongo-sft"
model.save_pretrained(save_dir)
tok.save_pretrained(save_dir)
print("保存しました")